文件是一个自定义的 Python 日志系统，提供了分级日志记录功能，包括调试、信息、强调、警告、错误和致命错误等不同级别的日志输出。它支持控制台彩色输出和日志文件记录，并提供了简洁的 API 设计。

与标准库对比
这个自定义日志系统比 Python 标准库 logging 更轻量，适合小型项目或快速原型开发。但缺少一些高级特性，如多处理器安全、日志轮转、复杂过滤等。如果项目规模变大，建议迁移到标准库或更成熟的第三方日志库。

In [ ]:
from datetime import datetime
import sys
import traceback

# 1.日志系统基础设置

# 日志级别
DEBUG = -1
INFO = 0
EMPH = 1
WARNING = 2
ERROR = 3
FATAL = 4

# 低于 INFO 级别的 DEBUG 日志会被忽略，只显示 INFO 及以上的日志
log_level = INFO

# 分隔线
line_seg = ''.join(['*'] * 65)


class LoggerFatalError(SystemExit):
    pass


def _format(level, messages):
    '''
    格式化日志消息为标准格式

    level: 日志级别标识（如 'I' 表示 INFO）
    messages: 日志内容（可变参数，如 ["操作成功", 200]）

    return: 格式化后的日志字符串
    '''

    # 生成时间戳：取当前时间并格式化为 月.日/时:分 的形式，例：06.10/16:45。
    timestr = datetime.strftime(datetime.now(), '%m.%d/%H:%M')

    # 获取调用位置信息
    # 假设在 main.py 的第 45 行调用了 info()，则 father 包含：
    # FrameSummary(filename='main.py', lineno=45, name='<module>', line=None)
    father = traceback.extract_stack()[-4]


    func_info = f'{father[0].split("/")[-1]}:{str(father[1]).ljust(4, " ")}'

    # 将用户传入的多个参数转换为字符串并拼接成一句话，例：
    # 输入 ["操作成功", 200] → 输出 "操作成功 200"
    m = ' '.join(map(str, messages))

    # 生成完整日志格式
    # level：日志级别标识（如 I 表示 INFO）。
    # timestr：时间戳。
    # func_info：文件名和行号（行号后有空格填充）。
    # m：用户传入的日志内容。
    # 最终格式：[级别] 时间戳 文件名:行号] 日志内容，例：
    # I 06.10/16:45 main.py:45 ] 服务器启动成功 端口: 8080
    msg = f'{level} {timestr} {func_info}] {m}'

    return msg


_log_file = None
_log_buffer = []
_RED = '\033[0;31m'
_GREEN = '\033[1;32m'
_LIGHT_RED = '\033[1;31m'
_ORANGE = '\033[0;33m'
_YELLOW = '\033[1;33m'
_NC = '\033[0m'  # No Color


def set_file(fname):
    global _log_file
    global _log_buffer
    if _log_file is not None:
        warning("Change log file to %s" % fname)
        _log_file.close()
    _log_file = open(fname, 'w')
    if len(_log_buffer):
        for s in _log_buffer:
            _log_file.write(s)
        _log_file.flush()


def debug(*messages, file=None):

    if log_level > DEBUG:
        return

    msg = _format('D', messages)


    if file is None:
        sys.stdout.write(_YELLOW + msg + _NC + '\n')
        sys.stdout.flush()

    else:
        with open(file, 'a+') as f:
            print(msg, file=f)


def info(*messages, file=None):
    '''
    *messages：可变参数，允许传入任意数量的参数（如 info("a", "b", 123)）。
    file=None：可选参数，指定日志输出的文件路径（默认输出到控制台）。
    '''

    # log_level：是全局变量，控制日志输出的最低级别
    # 如果log_level高于INFO，则忽略本次信息
    if log_level > INFO:
        return

    # 调用 _format 生成标准日志格式
    msg = _format('I', messages)

    # 默认控制台输出日志
    if file is None:
        sys.stdout.write(msg + '\n')
        sys.stdout.flush()

    # 制定路径输出日志
    else:
        # 追加模式（'a+'）打开文件，不存在则创建
        with open(file, 'a+') as f:
            # 将格式化后的日志写入文件（等价于 f.write(msg + '\n')）
            print(msg, file=f)


def emph(*messages, file=None):

    if log_level > EMPH:
        return

    msg = _format('EM', messages)

    if file is None:
        sys.stdout.write(_GREEN + msg + _NC + '\n')
        sys.stdout.flush()

    else:
        with open(file, 'a+') as f:
            print(msg, file=f)


def warning(*messages, file=None):

    if log_level > WARNING:
        return

    msg = _format('W', messages)

    if file is None:
        sys.stderr.write(_ORANGE + msg + _NC + '\n')
        sys.stderr.flush()

    else:
        with open(file, 'a+') as f:
            print(msg, file=f)


def error(*messages, file=None):

    if log_level > ERROR:
        return

    msg = _format('E', messages)

    if file is None:
        sys.stderr.write(_RED + msg + _NC + '\n')
        sys.stderr.flush()

    else:
        with open(file, 'a+') as f:
            print(msg, file=f)


def fatal(*messages, file=None):

    if log_level > FATAL:
        return

    msg = _format('F', messages)

    if file is None:
        sys.stderr.write(_LIGHT_RED + msg + _NC + '\n')
        sys.stderr.flush()

    else:
        with open(file, 'a+') as f:
            print(msg, file=f)

    raise LoggerFatalError(-1)